# Baseline Models for PTF Forecasting


In [23]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error , mean_squared_error




In [29]:
FEATURE_PATH = Path("../data/features/ptf_features.parquet")

df = pd.read_parquet(FEATURE_PATH)

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")

df = df.sort_index()
print(df.shape)
df.head()

(71304, 18)


,ptf,hour,day_of_week,dow_sin,dow_cos,is_weekend,hour_sin,hour_cos,lag_1,lag_24,lag_168,lag_336,rolling_mean_24,rolling_mean_168,diff_1,diff_24,diff_168,target
date,,,,,,,,,,,,,,,,,,
2018-01-15 00:00:00,193.13,0,0,0.0,1.0,0,0.000000,1.000000,174.00,179.81,173.74,207.60,178.616667,184.406607,-13.99,-9.40,10.77,189.40
2018-01-15 01:00:00,193.12,1,0,0.0,1.0,0,0.258819,0.965926,193.13,180.00,169.99,205.34,179.171667,184.522024,19.13,13.32,19.39,178.99
2018-01-15 02:00:00,125.01,2,0,0.0,1.0,0,0.500000,0.866025,193.12,159.99,135.00,164.94,179.718333,184.659702,-0.01,13.12,23.13,169.07
2018-01-15 03:00:00,125.01,3,0,0.0,1.0,0,0.707107,0.707107,125.01,185.00,134.99,154.52,178.260833,184.600238,-68.11,-34.98,-9.99,170.00
2018-01-15 04:00:00,125.01,4,0,0.0,1.0,0,0.866025,0.500000,125.01,160.00,134.99,112.64,175.761250,184.540833,0.00,-59.99,-9.98,160.00


In [30]:
TARGET_COL ="ptf"

baseline_cols = [
    "lag_1",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
    "rolling_mean_168",
]

split_idx = int(len(df)*0.8)

train_df = df.iloc[:split_idx].copy()
test_df =df.iloc[split_idx:].copy()

print("Train shape:",train_df.shape)
print("Test shape:",test_df.shape)

Train shape: (57043, 18)
Test shape: (14261, 18)


In [31]:
def mean_absolute_percentage_error_safe(y_true,y_pred):
    y_true =np.array(y_true)
    y_pred =np.array(y_pred)

    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate_forecast(y_true,y_pred):
    mae = mean_absolute_error(y_true,y_pred)
    rmse = np.sqrt(mean_squared_error(y_true,y_pred))
    mape = mean_absolute_percentage_error_safe(y_true,y_pred)

    return {
        "MAE":mae,
        "RMSE":rmse,
        "MAPE":mape,
    }



In [33]:
results =[]

y_test = test_df[TARGET_COL]

for col in baseline_cols:
    pred = test_df[col]

    valid_mask =y_test.notna() & pred.notna()

    metrics =evaluate_forecast(
        y_true = y_test[valid_mask],
        y_pred =pred[valid_mask]
    )

    metrics["Model"] = col
    results.append(metrics)

train_mean = train_df[TARGET_COL].mean()
mean_pred =pd.Series(train_mean,index=test_df.index)

metrics_mean = evaluate_forecast(test_df[TARGET_COL],mean_pred)
metrics_mean["Model"] ="train_mean"
results.append(metrics_mean)

results_df =pd.DataFrame(results)
results_df =results_df[["Model","MAE","RMSE","MAPE"]]
results_df = results_df.sort_values("MAE").reset_index(drop =True)
results_df


,Model,MAE,RMSE,MAPE
0,lag_1,268.425533,420.580993,19.428655
1,lag_168,386.444004,614.996588,109.567710
2,lag_24,389.505819,642.282220,164.205039
3,rolling_mean_24,489.867054,655.376070,264.916141
4,rolling_mean_168,520.902174,695.571085,345.167261
5,train_mean,1536.316387,1649.865565,204.161520
